# MASH preprocessing

## Description

Build MASH (Multivariate Adaptive Shrinkage) inputs from multi-context tensorqtl summary statistics and estimate the mixture-component covariance + weights via `pecotmr::mashPipeline()`. Three SoS steps:

- **`[generate_manifest]`** &mdash; calls `mash_manifest.R` to resolve `--region-file` + `--sum-files` + `--conditions` into a single per-region manifest TSV (one row per region, with chrom-matched per-condition tensorqtl paths already resolved). All region parsing / per-chrom matching lives in R.
- **`[mash_inputs_from_tensorqtl]`** &mdash; per-region fan-out. For each region in `--region-file`, calls `mash_sumstats_construct.R` to read the per-condition tensorqtl files, inner-join on `variant_id`, and write a per-region `list(region_id = list(z = <variants x conditions matrix>))` RDS.
- **`[mash_estimation]`** &mdash; concatenates every per-region RDS, runs `pecotmr::mashRandNullSample` to derive strong / random / null subsets, wraps each partition as a per-context `QtlSumStats`, and calls `pecotmr::mashPipeline()`. The output RDS carries the MASH `U` (covariance matrices) and `w` (mixture weights).

Replaces the workflow in `multivariate_genome/MASH/mash_preprocessing.ipynb` (which depended on the now-deprecated `load_multitrait_*_sumstat` loaders). The legacy notebook is left in place; new work should use this one.

## Inputs

- `--region-file` &mdash; TSV with columns `chr start end gene_id` listing the LD-block regions to process.
- `--sum-files file1 [file2 ...]` &mdash; text files, one per condition; each contains one tensorqtl summary-statistics path per line. The notebook assumes the per-chromosome layout already used by `mash_ran_null_sample` callers (path filename embeds the chromosome, matching the legacy `.CHR.` convention).
- `--conditions c1,c2,...` &mdash; condition labels, one per `--sum-files` entry (same order).
- `--ld-sketch <RDS>` &mdash; `GenotypeHandle` RDS (required by `mashPipeline()`'s QC gate; the synthesised `QtlSumStats` carries it through `summaryStatsQc()`).
- `--n-random / --n-null` &mdash; subset sizes for `mashRandNullSample`. Defaults 4000 / 4000.
- `--exclude-condition c1,c2` &mdash; conditions to drop before MASH. Default none.
- `--alpha` &mdash; alpha forwarded to `mashPipeline()`. Default 0 (standard scale).
- `--seed` &mdash; RNG seed. Default 999.
- `--name` &mdash; output filename prefix.

## Outputs

- `{cwd}/{name}_cache/{name}.{gene_id}.mash_input.rds` &mdash; one per region.
- `{cwd}/{name}.mash_result.rds` &mdash; final `mashPipeline()` output (`list(U, w)`).

## MWE note

The legacy fixture in `input/finemapping/protocol_example.sumstats_db.rds` was custom-massaged; the new pipeline expects standard per-condition tensorqtl outputs plus a `GenotypeHandle` RDS for the LD reference. Provide both before running.

## Example

```bash
sos run pipeline/mash_preprocessing.ipynb \
    mash_inputs_from_tensorqtl+mash_estimation \
    --cwd output/mash --modular-script-dir /path/to/code/script \
    --name protocol_example_mash \
    --region-file input/finemapping/protocol_example.region \
    --sum-files input/mash/condA.txt input/mash/condB.txt \
    --conditions condA,condB \
    --ld-sketch input/ld/protocol_example.ld_sketch.rds \
    --n-random 4000 --n-null 4000 \
    --alpha 0 --seed 999
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: modular_script_dir = path('code/script')
parameter: name = str
# --- per-region input ----------------------------------------------
parameter: region_file = path
parameter: sum_files = paths     # one txt per condition; each line is one tensorqtl file path
parameter: conditions = ''       # comma-separated condition labels
# --- mash estimation knobs ------------------------------------------
parameter: ld_sketch = path
parameter: n_random = 4000
parameter: n_null   = 4000
parameter: exclude_condition = ''  # comma-separated
parameter: alpha = 0.0
parameter: seed  = 999
# --- infrastructure -------------------------------------------------
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '2h'
parameter: mem = '16G'
parameter: numThreads = 1

In [ ]:
[generate_manifest]
# Resolve --region-file x --sum-files x --conditions into one manifest
# TSV via mash_manifest.R. Downstream steps fan out over its rows; no
# Python parsing in this notebook.
input: None
output: f"{cwd}/{name}.mash_manifest.tsv"
task: trunk_workers = 1, trunk_size = 1, walltime = '15m', mem = '2G', cores = 1, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/mash_manifest.R \
        --region-file ${region_file} \
        --sum-files ${' '.join(str(x) for x in sum_files)} \
        --conditions ${conditions} \
        --output ${_output}

In [ ]:
[mash_inputs_from_tensorqtl]
# Fan out over the manifest's rows; one per-region task each.
import csv
jobs = list(csv.DictReader(open(f"{cwd}/{name}.mash_manifest.tsv"), delimiter='\t'))
input: for_each = 'jobs'
output: f"{cwd}/{name}_cache/{name}.{_jobs['gene_id']}.mash_input.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/mash_sumstats_construct.R \
        --tensorqtl-paths ${_jobs['tensorqtl_paths']} \
        --conditions ${conditions} \
        --region ${_jobs['region']} \
        --output ${_output}

In [ ]:
[mash_estimation]
output: f"{cwd}/{name}.mash_result.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/mash.R \
        --mash-inputs ${_input} \
        --study ${name} \
        --ld-sketch ${ld_sketch} \
        --n-random ${n_random} \
        --n-null ${n_null} \
        ${('--exclude-condition ' + exclude_condition) if exclude_condition else ''} \
        --alpha ${alpha} \
        --seed ${seed} \
        --output ${_output}